# Prediction versus Causation and Potential Outcomes Lab


## Purpose

This lab introduces potential outcomes and shows why observed associations are not automatically causal effects.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 5000

data = pd.DataFrame({
    "unit_id": [f"unit_{i:05d}" for i in range(1, n + 1)],
    "prior_activity": rng.normal(0, 1, size=n),
    "domain_expertise": rng.normal(0, 1, size=n),
})

data["true_tau"] = (
    0.08
    + 0.04 * (data["prior_activity"] > 0)
    + 0.03 * (data["domain_expertise"] > 0)
)

data["y0"] = (
    0.30
    + 0.08 * data["prior_activity"]
    + 0.04 * data["domain_expertise"]
    + rng.normal(0, 0.05, size=n)
)

data["y1"] = data["y0"] + data["true_tau"]

data.head()

In [ ]:
true_ate = data["true_tau"].mean()

data["confounded_treatment"] = rng.binomial(
    1,
    1 / (1 + np.exp(-(-0.2 + 1.2 * data["prior_activity"]))),
)

data["observed_outcome"] = np.where(
    data["confounded_treatment"] == 1,
    data["y1"],
    data["y0"],
)

naive_difference = (
    data.loc[data["confounded_treatment"] == 1, "observed_outcome"].mean()
    - data.loc[data["confounded_treatment"] == 0, "observed_outcome"].mean()
)

pd.DataFrame([
    {"quantity": "true_ate", "value": true_ate},
    {"quantity": "naive_observed_difference", "value": naive_difference},
])

## Interpretation

The naive observed difference can be biased when treatment assignment depends on pre-treatment covariates.